# 03 — On-sky visit coverage per night

ConsDB `cdb_lsstcam.visit1` with `airmass > 0` (keeps on-sky engineering, drops bias/dark/flat/CBP). Visit intervals are
unioned after bridging gaps ≤ `gap_min`; longer gaps, including the two edge gaps against the −18° limits, are loss windows.
`gap_min` = 10 min (Bruno, 2026-10-18); sensitivity at 2 / 5 / 10 min is reported. ConsDB MJDs are TAI and are converted
to UTC.

In [ ]:
import sys, logging
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, "..")
from time_loss import DAY_OBS_MIN, DAY_OBS_MAX, GAP_MIN_MINUTES, GAP_MIN_SENSITIVITY, DAY_OBS_WEATHER_BIT_START
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 300); pd.set_option("display.max_colwidth", 80)
logging.basicConfig(level=logging.WARNING)
for name in ("httpx", "numexpr", "time_loss.access"):
    logging.getLogger(name).setLevel(logging.WARNING)
STEP0 = Path("../data/step0"); CACHE = Path("../data/cache"); PROC = Path("../data/processed")
for p in (STEP0, CACHE, PROC): p.mkdir(parents=True, exist_ok=True)

In [ ]:
from time_loss import access, visits as V
REFRESH = False
vpath = STEP0 / "visits_onsky_20251025_20260714.parquet"
if REFRESH or not vpath.exists():
    visits = V.fetch_visits(access.get_consdb_client()); visits.to_parquet(vpath)
visits = pd.read_parquet(vpath)
limits = pd.read_csv(STEP0 / "night_limits.csv")
print(len(visits), "on-sky visits on", visits.day_obs.nunique(), "nights")
visits.img_type.value_counts()

In [ ]:
tabs = {g: V.coverage_table(visits, limits, gap_min=g)[0] for g in GAP_MIN_SENSITIVITY}
sens = pd.DataFrame({f"on_sky_h_gap{int(g)}": t.on_sky_hours for g, t in tabs.items()})
sens.insert(0, "day_obs", limits.day_obs)
cov_tab, cov = V.coverage_table(visits, limits, gap_min=GAP_MIN_MINUTES)
out = cov_tab.merge(sens, on="day_obs").merge(limits[["day_obs", "night_hours_18", "night_hours_12"]], on="day_obs")
out.to_csv(STEP0 / "visit_coverage_gapmin_2_5_10.csv", index=False)
import pickle
with open(CACHE / "visit_coverage.pkl", "wb") as fh:
    pickle.dump(cov, fh)
print("night_hours_12 total (accounting night, ROSP-22):", round(limits.night_hours_12.sum(), 1), "| night_hours_18:", round(limits.night_hours_18.sum(), 1))
sens.drop(columns="day_obs").sum().round(1)

## Spot checks against the night reports

In [ ]:
checks = {20251031: "report: on sky only the last ~1.5 h (humidity, clouds)", 20260307: "report: ~3 h observed, ≈7 h lost",
          20260413: "report: ~5.5 h observed (thick clouds)", 20260531: "report: ≈3.3 h on sky then clouds", 20260626: "report: ~11.5 h, no faults"}
o = out.set_index("day_obs")
for d, note in checks.items():
    r = o.loc[d]; print(f"{d}: on_sky {r.on_sky_hours:.2f} h of {r.night_hours_12:.2f} h night (−12°), {int(r.n_visits)} visits, {int(r.n_gaps)} gaps — {note}")

In [ ]:
import matplotlib.pyplot as plt
from time_loss.style import COLORS
fig, ax = plt.subplots(figsize=(10, 3.5))
x = pd.to_datetime(out.day_obs.astype(str))
ax.bar(x, out.night_hours_12, color=COLORS["grid"], width=1, label="−12° night")
ax.bar(x, out.on_sky_hours, color=COLORS["on_sky"], width=1, label=f"on sky (gap_min {int(GAP_MIN_MINUTES)} min)")
ax.set_ylabel("hours"); ax.legend(frameon=False, loc="upper left"); fig.tight_layout()